<a href="https://colab.research.google.com/github/andandandand/practical-computer-vision/blob/feat/nemotron-parse-2-britannica-colab/notebooks/nemotron_parse_2_britannica_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


<img src="https://developer.download.nvidia.com/compute/machine-learning/frameworks/nvidia_logo.png" align="right" width="100px"/>

# Historical document intelligence with Nemotron Parse 2.0 + Nemotron 3 Nano Omni

This short Colab turns **five pages selected from the Britannica Illustrated Pages dataset** into grounded, cross-page evidence. It uses hosted NVIDIA endpoints, so a **CPU runtime is sufficient**.

| Pipeline role | Hosted model | Work in this demo |
| --- | --- | --- |
| Spatial parser | `nvidia/nemotron-parse-2.0` | 5 page calls |
| Visual specialist | `nvidia/nemotron-3-nano-omni-30b-a3b-reasoning` | up to 5 crop calls |
| Reasoning engine | same Nano Omni endpoint | 1 text-only call |

Normal maximum: **11 model calls**. Status polling for an accepted NVIDIA request does not create another inference request.

Model pages: [Nemotron Parse 2.0](https://build.nvidia.com/nvidia/nemotron-parse-2.0) · [Nemotron 3 Nano Omni](https://build.nvidia.com/nvidia/nemotron-3-nano-omni-30b-a3b-reasoning)


## 1. What the notebook demonstrates

```text
Britannica page  ──►  Parse 2.0 blocks + bboxes
                          │
                          └──► one selected visual crop ──► Nano Omni transcription

five spatially ordered page records ──► one grounded cross-page answer
```

You will:

1. authenticate through Colab Secrets;
2. retrieve five pinned pages from [`biglam/britannica-illustrated-pages`](https://huggingface.co/datasets/biglam/britannica-illustrated-pages);
3. inspect Parse 2.0 classes, text, reading order, and bounding boxes;
4. enrich at most one visual region per page with Nano Omni; and
5. ask one editable question over the assembled evidence.

**Evidence boundary.** Britannica's `p_illustrated` field and optional crop masks are model predictions, not human ground truth. This notebook performs qualitative inspection; it does not report those predictions as Parse 2.0 accuracy. Nano Omni crop descriptions are also clearly marked as generated visual interpretations, never verbatim OCR.


## 2. Setup

Colab already provides `requests`, Pillow, pandas, and IPython. We install the current Hugging Face Hub client used to verify the pinned dataset revision.


In [ ]:
!pip install --quiet --upgrade "huggingface_hub>=0.34"
print("[setup] Runtime dependencies are ready.")


### 2.1 Add two Colab Secrets

In the left sidebar, open the **key** icon, add these secrets, and enable **Notebook access** for both:

- `NVIDIA_API_KEY` — generated from [build.nvidia.com](https://build.nvidia.com/nvidia/nemotron-parse-2.0)
- `HF_TOKEN` — a Hugging Face read token

Secrets are read only at runtime and are never written to notebook output or result files.


In [ ]:
from google.colab import userdata


try:
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "A required Colab Secret is missing or notebook access is disabled. "
        "Open the key sidebar, add NVIDIA_API_KEY and HF_TOKEN, enable Notebook access, "
        "and rerun this cell."
    ) from exc

for name, value in {"NVIDIA_API_KEY": NVIDIA_API_KEY, "HF_TOKEN": HF_TOKEN}.items():
    if not value:
        raise RuntimeError(f"Colab Secret {name!r} is empty.")
print("[auth] NVIDIA_API_KEY and HF_TOKEN loaded from Colab Secrets.")


## 3. Configuration and the five-page manifest

The manifest is versioned at [`assets/nemotron-parse-2-britannica/pages.json`](https://github.com/andandandand/practical-computer-vision/blob/feat/nemotron-parse-2-britannica-colab/assets/nemotron-parse-2-britannica/pages.json). It is embedded below so the notebook remains runnable even if a branch or raw-file URL changes.

Each stable `[B1]`–`[B5]` label carries the dataset `file_key`, page provenance, a 2000-pixel Hugging Face image URL, its expected SHA-256 digest, and the reason that page was chosen.


In [ ]:
from __future__ import annotations

import base64
import hashlib
import io
import json
import re
import time
from collections import Counter
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from huggingface_hub import HfApi
from IPython.display import Markdown, display
from PIL import Image, ImageDraw, ImageFont, ImageOps

MANIFEST = json.loads(r"""{"schema_version": 2, "dataset_id": "biglam/britannica-illustrated-pages", "dataset_config": "pages", "dataset_split": "train", "dataset_revision": "d14ad4cf2717f97b63c668c3d6a83e6257791e33", "dataset_card_url": "https://huggingface.co/datasets/biglam/britannica-illustrated-pages", "selection_policy": "Five visually verified pages spanning distinct historical document-parsing challenges. file_key is the stable identity; row_idx is retained only as a verification aid for this pinned revision. source_sha256 pins the exact bytes fetched for this demo.", "evaluation_notice": "p_illustrated and crop-mask fields are model predictions, not human ground truth. This manifest supports a qualitative demo, not an accuracy benchmark.", "pages": [{"label": "B1", "challenge": "Dense text with an embedded illustration", "selection_rationale": "A dense two-column GAS / MANUFACTURE article with a regenerative-furnace figure embedded in the reading flow.", "preferred_region_types": ["Picture", "Chart", "Table", "Figure"], "row_idx": 100033, "file_key": "pages/text-figs/encyclopaedia-britannica-11th-and-12th-editions/Encyclopædia Britannica - Volume 11_p0403.jpg", "edition": "12th", "year": 1922, "page": 512, "title": "Encyclopædia Britannica 11th and 12th Editions", "contributor": "", "ia_url": "https://archive.org/details/encyclopaedia-britannica-11th-and-12th-editions", "ia_page_url": "https://archive.org/details/encyclopaedia-britannica-11th-and-12th-editions/Encyclop%C3%A6dia%20Britannica%20-%20Volume%2011/page/n511/mode/1up", "bucket_url_jpg": "https://huggingface.co/buckets/biglam/britannica/resolve/source/jpg/encyclopaedia-britannica-11th-and-12th-editions/Encyclopædia Britannica - Volume 11_0511.jpg", "source_sha256": "899b27b5c674c2efed87fbf0b9a56a06797c24c3e6b4c557a6194c68e11180fd", "source_bytes": 987111, "words": 1820, "stratum": "text", "p_illustrated": 0.9997919201850891}, {"label": "B2", "challenge": "Full-page engraved plate", "selection_rationale": "A full-page ARACHNIDES scientific engraving containing multiple specimens and printed captions.", "preferred_region_types": ["Picture", "Figure", "Chart", "Table"], "row_idx": 20010, "file_key": "pages/plates/encyclopaedia-britannica-7ed-1842/Vol 3 (Anatomy-Astronomy) 193060122.23_p0938.jp2", "edition": "7th", "year": 1842, "page": 863, "title": "Encyclopaedia Britannica 7th Edition 1842", "contributor": "", "ia_url": "https://archive.org/details/encyclopaedia-britannica-7ed-1842", "ia_page_url": "https://archive.org/details/encyclopaedia-britannica-7ed-1842/Vol%203%20%28Anatomy-Astronomy%29%20193060122.23/page/n862/mode/1up", "bucket_url_jpg": "https://huggingface.co/buckets/biglam/britannica/resolve/source/jpg/encyclopaedia-britannica-7ed-1842/Vol 3 (Anatomy-Astronomy) 193060122.23_0862.jpg", "source_sha256": "f916cfde042eadd32910485dde9b84c7c3f5115a645f84c61dfb94ae1d5edae3", "source_bytes": 345799, "words": 32, "stratum": "low", "p_illustrated": 0.9998121857643127}, {"label": "B3", "challenge": "Structured numerical tables", "selection_rationale": "An IOWA article page with several ruled numerical tables covering crops, livestock, railways, and population.", "preferred_region_types": ["Table", "Chart", "Picture", "Figure"], "row_idx": 110003, "file_key": "pages/text-figs/cu31924032607784/cu31924032607784_p0393.jpg", "edition": "9th", "year": 1878, "page": 476, "title": "The Encyclopædia Britannica; a dictionary of arts, sciences, and general literature", "contributor": "Cornell University Library", "ia_url": "https://archive.org/details/cu31924032607784", "ia_page_url": "https://archive.org/details/cu31924032607784/cu31924032607784/page/n475/mode/1up", "bucket_url_jpg": "https://huggingface.co/buckets/biglam/britannica/resolve/source/jpg/cu31924032607784/cu31924032607784_0475.jpg", "source_sha256": "77dbdddfe638c9951a4525e41b5b95054b0fa142c49d33e82e62e5cee9878858", "source_bytes": 754234, "words": 955, "stratum": "text", "p_illustrated": 0.9800528883934021}, {"label": "B4", "challenge": "Labeled architectural diagram", "selection_rationale": "A full-page Tuscan-order architectural plate with labels, dimension lines, and mixed horizontal and vertical text.", "preferred_region_types": ["Picture", "Figure", "Chart", "Table"], "row_idx": 75006, "file_key": "pages/plates/encyclopaediabri0000vari_t8n7/encyclopaediabri0000vari_t8n7_p0410.jp2", "edition": "1st", "year": 1771, "page": 410, "title": "Encyclopaedia Britannica; Or, A Dictionary Of Arts And Science", "contributor": "Internet Archive", "ia_url": "https://archive.org/details/encyclopaediabri0000vari_t8n7", "ia_page_url": "https://archive.org/details/encyclopaediabri0000vari_t8n7/encyclopaediabri0000vari_t8n7/page/n410/mode/1up", "bucket_url_jpg": "https://huggingface.co/buckets/biglam/britannica/resolve/source/jpg/encyclopaediabri0000vari_t8n7/encyclopaediabri0000vari_t8n7_0410.jpg", "source_sha256": "44cb380d461b9c60d538e1ea89239fcbe7bf54e54d40375bfe97bf1d8d46f5cb", "source_bytes": 296740, "words": 68, "stratum": "low", "p_illustrated": 0.9995604157447815}, {"label": "B5", "challenge": "Unusual historical typography", "selection_rationale": "A German Fraktur / blackletter page derived from the eighth-edition Britannica, selected as a visibly different OCR challenge.", "preferred_region_types": ["Picture", "Chart", "Table", "Figure"], "row_idx": 50028, "file_key": "pages/text-figs/bub_gb_ack-AAAAcAAJ/bub_gb_ack-AAAAcAAJ_p0042.jpg", "edition": "8th", "year": 1856, "page": 174, "title": "Die englischen Pendeluhren, u. zwar Thurmuhren, Hausuhren, Controluhren u. astronomische Regulatoren, sowie Taschenuhren u. Chronometer, mit der neuesten Verbesserungen Nach der 8. Aufl. der Encyclopaedia Britannica übers. Neuer Schauplatz d. Künste u. Handwerke. 9te Bd.", "contributor": "Bavarian State Library", "ia_url": "https://archive.org/details/bub_gb_ack-AAAAcAAJ", "ia_page_url": "https://archive.org/details/bub_gb_ack-AAAAcAAJ/bub_gb_ack-AAAAcAAJ/page/n173/mode/1up", "bucket_url_jpg": "https://huggingface.co/buckets/biglam/britannica/resolve/source/jpg/bub_gb_ack-AAAAcAAJ/bub_gb_ack-AAAAcAAJ_0173.jpg", "source_sha256": "ee31d49d55e8aaf156550dbdac30e802a443c6de4b4dd417517c4abb495ce437", "source_bytes": 258451, "words": 163, "stratum": "text", "p_illustrated": 0.9482765793800354}]}""")
DEMO_PAGES = MANIFEST["pages"]
DATASET_ID = MANIFEST["dataset_id"]
DATASET_REVISION = MANIFEST["dataset_revision"]

NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"
PARSE_MODEL = "nvidia/nemotron-parse-2.0"
NANO_OMNI_MODEL = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning"
PARSE_PROMPT = "</s><s><predict_bbox><predict_classes><output_markdown><predict_no_text_in_pic>"

CALL_COUNTS = {"parse": 0, "enrichment": 0, "qa": 0}
MAX_PARSE_CALLS = 5
MAX_ENRICHMENT_CALLS = 5
MAX_QA_CALLS = 1

assert [page["label"] for page in DEMO_PAGES] == ["B1", "B2", "B3", "B4", "B5"]
assert len({page["file_key"] for page in DEMO_PAGES}) == 5

hf_info = HfApi(token=HF_TOKEN).dataset_info(DATASET_ID, revision=DATASET_REVISION)
print(f"[dataset] {DATASET_ID} @ {hf_info.sha[:12]} verified with HF_TOKEN")
print(f"[models]  {PARSE_MODEL} + {NANO_OMNI_MODEL}")


In [ ]:
manifest_table = pd.DataFrame([
    {
        "label": p["label"],
        "challenge": p["challenge"],
        "edition": p["edition"],
        "year": p["year"],
        "page": p["page"],
        "words": p["words"],
        "file_key": p["file_key"],
    }
    for p in DEMO_PAGES
])
display(manifest_table)


## 4. Small, explicit helpers

The helpers below handle four boundaries: authenticated Hugging Face downloads, image sizing, NVIDIA's optional asynchronous response, and Parse 2.0's coordinate-token output.

Parse 2.0 resizes and center-pads pages internally. The bbox conversion mirrors NVIDIA's published postprocessor so overlays remain aligned on portrait and landscape scans.


In [ ]:
_HF_SESSION = requests.Session()
_HF_SESSION.headers.update({"Authorization": f"Bearer {HF_TOKEN}"})

_NVIDIA_HEADERS = {
    "Authorization": f"Bearer {NVIDIA_API_KEY}",
    "Accept": "application/json",
    "Content-Type": "application/json",
}


def download_page(record: dict[str, Any]) -> Image.Image:
    """Download one pinned 2000 px page from the Hugging Face bucket."""
    response = _HF_SESSION.get(record["bucket_url_jpg"], timeout=90)
    response.raise_for_status()
    payload = response.content
    actual_sha256 = hashlib.sha256(payload).hexdigest()
    if actual_sha256 != record["source_sha256"]:
        raise RuntimeError(
            f"SHA-256 mismatch for {record['label']}: expected "
            f"{record['source_sha256']}, received {actual_sha256}."
        )
    image = Image.open(io.BytesIO(payload)).convert("RGB")
    image.load()
    return image


def fit_for_parse(image: Image.Image) -> Image.Image:
    """Preserve aspect ratio inside Parse 2.0's recommended maximum frame."""
    return ImageOps.contain(image.convert("RGB"), (1664, 2048), Image.Resampling.LANCZOS)


def image_data_url(image: Image.Image, *, quality: int = 88) -> str:
    buffer = io.BytesIO()
    image.convert("RGB").save(buffer, format="JPEG", quality=quality, optimize=True)
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return f"data:image/jpeg;base64,{encoded}"


def nvidia_completion(payload: dict[str, Any], *, timeout: int = 300) -> dict[str, Any]:
    """Submit once, then poll the same NVIDIA request when it returns HTTP 202."""
    response = requests.post(
        f"{NVIDIA_BASE_URL}/chat/completions",
        headers=_NVIDIA_HEADERS,
        json=payload,
        timeout=timeout,
    )
    deadline = time.monotonic() + timeout
    while response.status_code == 202:
        request_id = response.headers.get("NVCF-REQID")
        if not request_id:
            raise RuntimeError("NVIDIA returned 202 without an NVCF-REQID header.")
        if time.monotonic() >= deadline:
            raise TimeoutError(f"NVIDIA request {request_id} did not complete in {timeout}s.")
        time.sleep(2)
        response = requests.get(
            f"{NVIDIA_BASE_URL}/status/{request_id}",
            headers=_NVIDIA_HEADERS,
            timeout=60,
        )
    response.raise_for_status()
    body = response.json()
    choice = (body.get("choices") or [{}])[0]
    if choice.get("finish_reason") == "length":
        raise RuntimeError("The model response hit its token limit; extraction may be incomplete.")
    return body


def message_text(body: dict[str, Any]) -> str:
    message = (body.get("choices") or [{}])[0].get("message") or {}
    content = message.get("content") or ""
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        return "\n".join(
            part.get("text", "") for part in content if isinstance(part, dict)
        ).strip()
    return str(content).strip()


def visible_answer(raw: str) -> str:
    """Hide a delimited thinking trace while retaining raw output in results."""
    if "</think>" in raw:
        return raw.rsplit("</think>", 1)[-1].strip()
    return re.sub(r"<think\b[^>]*>.*?</think>", "", raw, flags=re.DOTALL | re.IGNORECASE).strip()


def json_object_from_text(raw: str) -> dict[str, Any]:
    cleaned = visible_answer(raw).strip()
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", cleaned, flags=re.IGNORECASE)
    try:
        value = json.loads(cleaned)
        return value if isinstance(value, dict) else {}
    except json.JSONDecodeError:
        left, right = cleaned.find("{"), cleaned.rfind("}")
        if left >= 0 and right > left:
            try:
                value = json.loads(cleaned[left:right + 1])
                return value if isinstance(value, dict) else {}
            except json.JSONDecodeError:
                pass
    return {}


In [ ]:
_BLOCK_RE = re.compile(
    r"<x_(\d+(?:\.\d+)?)><y_(\d+(?:\.\d+)?)>(.*?)"
    r"<x_(\d+(?:\.\d+)?)><y_(\d+(?:\.\d+)?)><class_([^>]+)>",
    re.DOTALL,
)


def bbox_to_page(
    bbox: tuple[float, float, float, float],
    page_width: int,
    page_height: int,
    *,
    target_width: int = 1664,
    target_height: int = 2048,
) -> list[float]:
    """Undo Parse 2.0's aspect-preserving resize and centered padding."""
    aspect = page_width / page_height
    resized_width, resized_height = page_width, page_height
    if resized_height > target_height:
        resized_height = target_height
        resized_width = int(resized_height * aspect)
    if resized_width > target_width:
        resized_width = target_width
        resized_height = int(resized_width / aspect)

    pad_left = max(0, target_width - resized_width) // 2
    pad_top = max(0, target_height - resized_height) // 2
    x0, y0, x1, y1 = bbox
    left = ((x0 * target_width) - pad_left) * page_width / resized_width
    right = ((x1 * target_width) - pad_left) * page_width / resized_width
    top = ((y0 * target_height) - pad_top) * page_height / resized_height
    bottom = ((y1 * target_height) - pad_top) * page_height / resized_height
    return [
        max(0.0, min(page_width, left)),
        max(0.0, min(page_height, top)),
        max(0.0, min(page_width, right)),
        max(0.0, min(page_height, bottom)),
    ]


def parse_blocks(raw: str, image: Image.Image) -> list[dict[str, Any]]:
    blocks: list[dict[str, Any]] = []
    for index, match in enumerate(_BLOCK_RE.finditer(raw)):
        x0, y0, text, x1, y1, class_name = match.groups()
        class_name = "Formula" if class_name == "Inline-formula" else class_name
        clean_text = (
            text.replace("<tbc>", "")
            .replace(r"\<|unk|\>", "")
            .replace(r"\unknown", "")
            .strip()
        )
        model_bbox = tuple(float(v) for v in (x0, y0, x1, y1))
        page_bbox = bbox_to_page(model_bbox, image.width, image.height)
        if page_bbox[2] <= page_bbox[0] or page_bbox[3] <= page_bbox[1]:
            continue
        blocks.append({
            "id": index,
            "reading_order": index,
            "type": class_name,
            "text": clean_text,
            "bbox_model": list(model_bbox),
            "bbox_px": page_bbox,
        })
    return blocks


CLASS_COLORS = {
    "Title": "#D32F2F", "Section-header": "#E91E63", "Text": "#388E3C",
    "List-item": "#1976D2", "Caption": "#546E7A", "Table": "#00ACC1",
    "Chart": "#8E24AA", "Picture": "#6D4C41", "Figure": "#6D4C41",
    "Formula": "#FB8C00", "Page-header": "#757575", "Page-footer": "#757575",
}


def draw_overlay(
    image: Image.Image,
    blocks: list[dict[str, Any]],
    *,
    selected_id: int | None = None,
) -> Image.Image:
    output = image.copy()
    draw = ImageDraw.Draw(output)
    font = ImageFont.load_default()
    width = max(2, image.width // 650)
    for block in blocks:
        box = block["bbox_px"]
        selected = block["id"] == selected_id
        color = "#FF00AA" if selected else CLASS_COLORS.get(block["type"], "#616161")
        draw.rectangle(box, outline=color, width=width * (3 if selected else 1))
        label = f"{block['id']}:{block['type']}"
        x, y = int(box[0]), max(0, int(box[1]) - 14)
        draw.rectangle((x, y, x + max(44, len(label) * 7), y + 14), fill=color)
        draw.text((x + 2, y + 1), label, fill="white", font=font)
    return output


def area_ratio(block: dict[str, Any], image: Image.Image) -> float:
    x0, y0, x1, y1 = block["bbox_px"]
    return max(0.0, x1 - x0) * max(0.0, y1 - y0) / (image.width * image.height)


def choose_visual_block(
    blocks: list[dict[str, Any]],
    image: Image.Image,
    preferred_types: list[str],
) -> dict[str, Any] | None:
    """Choose at most one meaningful visual block using manifest-guided class priority."""
    rank = {name: len(preferred_types) - i for i, name in enumerate(preferred_types)}
    candidates = [
        block for block in blocks
        if block["type"] in rank and area_ratio(block, image) >= 0.005
    ]
    if not candidates:
        return None
    return max(candidates, key=lambda block: (rank[block["type"]], area_ratio(block, image)))


def crop_block(image: Image.Image, block: dict[str, Any], *, margin: float = 0.02) -> Image.Image:
    x0, y0, x1, y1 = block["bbox_px"]
    dx, dy = (x1 - x0) * margin, (y1 - y0) * margin
    box = (
        max(0, int(x0 - dx)), max(0, int(y0 - dy)),
        min(image.width, int(x1 + dx)), min(image.height, int(y1 + dy)),
    )
    return image.crop(box)


## 5. Hosted model adapters

Parse uses NVIDIA's documented control-token prompt. Nano Omni receives one crop and returns one compact JSON object that combines classification and transcription—there is no hidden second enrichment call or automatic retry.


In [ ]:
def call_parse_2(image: Image.Image) -> tuple[str, list[dict[str, Any]], dict[str, Any]]:
    if CALL_COUNTS["parse"] >= MAX_PARSE_CALLS:
        raise RuntimeError("Parse call budget exhausted.")
    CALL_COUNTS["parse"] += 1
    payload = {
        "model": PARSE_MODEL,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": PARSE_PROMPT},
                {"type": "image_url", "image_url": {"url": image_data_url(image)}},
            ],
        }],
        "max_tokens": 8192,
        "temperature": 0,
        "stream": False,
    }
    body = nvidia_completion(payload)
    raw = message_text(body)
    blocks = parse_blocks(raw, image)
    if not blocks:
        raise RuntimeError("Parse 2.0 returned no decodable class/bbox blocks.")
    return raw, blocks, body.get("usage") or {}


ENRICHMENT_PROMPT = """\
Inspect this single region cropped from a historical Encyclopaedia Britannica page.
Return exactly one JSON object with these keys:
- image_type: one of Chart, Table, Diagram, Map, Engraving, Photograph, Other
- sub_type: a short, specific label
- subject_matter: one factual sentence about what is visibly depicted
- transcription: preserve every legible title, label, number, legend, and relationship;
  use Markdown structure where useful and say when text is illegible
- uncertainty: a short note listing anything unclear, or an empty string

Do not infer facts that are not visible. Do not add a preamble or reasoning trace.
"""


def call_omni_enrichment(crop: Image.Image) -> tuple[dict[str, Any], str, dict[str, Any]]:
    if CALL_COUNTS["enrichment"] >= MAX_ENRICHMENT_CALLS:
        raise RuntimeError("Nano Omni enrichment call budget exhausted.")
    CALL_COUNTS["enrichment"] += 1
    payload = {
        "model": NANO_OMNI_MODEL,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": ENRICHMENT_PROMPT},
                {"type": "image_url", "image_url": {"url": image_data_url(crop)}},
            ],
        }],
        "max_tokens": 1200,
        "temperature": 0.2,
        "chat_template_kwargs": {"enable_thinking": False},
        "stream": False,
    }
    body = nvidia_completion(payload)
    raw = message_text(body)
    parsed = json_object_from_text(raw)
    if not parsed:
        parsed = {
            "image_type": "Unknown",
            "sub_type": "Unparsed response",
            "subject_matter": "",
            "transcription": visible_answer(raw),
            "uncertainty": "The response was not valid JSON; preserved as generated text.",
        }
    return parsed, raw, body.get("usage") or {}


## 6. Run the bounded five-page pipeline

Each page has its own error boundary. A failed download, Parse call, or enrichment is recorded as a warning while the remaining pages continue. The notebook never retries an inference automatically, which keeps the model-call ceiling honest.


In [ ]:
page_images: dict[str, Image.Image] = {}
selected_crops: dict[str, Image.Image] = {}
results: dict[str, dict[str, Any]] = {}

for record in DEMO_PAGES:
    label = record["label"]
    display(Markdown(
        f"### [{label}] {record['challenge']}  \n"
        f"**{record['edition']} edition · {record['year']} · dataset page {record['page']}**  \n"
        f"[Internet Archive source]({record['ia_page_url']})"
    ))
    result: dict[str, Any] = {
        "label": label,
        "source": {key: value for key, value in record.items() if key != "preferred_region_types"},
        "parse_model": PARSE_MODEL,
        "omni_model": NANO_OMNI_MODEL,
        "download_status": "pending",
        "parse_status": "not_run",
        "enrichment_status": "not_run",
        "warnings": [],
        "blocks": [],
    }
    results[label] = result

    try:
        original = download_page(record)
        page = fit_for_parse(original)
        page_images[label] = page
        result["download_status"] = "ok"
        result["source_image_size"] = list(original.size)
        result["model_input_size"] = list(page.size)
        preview_width = 430
        display(page.resize((preview_width, int(preview_width * page.height / page.width))))
    except Exception as exc:
        result["download_status"] = "error"
        result["warnings"].append(f"download: {type(exc).__name__}: {exc}")
        print(f"[{label}] download failed; continuing: {exc}")
        continue

    try:
        started = time.monotonic()
        raw_parse, blocks, usage = call_parse_2(page)
        result["parse_status"] = "ok"
        result["parse_seconds"] = round(time.monotonic() - started, 2)
        result["parse_usage"] = usage
        result["raw_parse_output"] = raw_parse
        result["blocks"] = blocks
        selected = choose_visual_block(
            blocks, page, record.get("preferred_region_types", ["Chart", "Table", "Picture", "Figure"])
        )
        result["selected_block_id"] = selected["id"] if selected else None

        counts = Counter(block["type"] for block in blocks)
        print(
            f"[{label}] Parse 2.0: {len(blocks)} blocks in {result['parse_seconds']:.1f}s · "
            f"classes={dict(counts)}"
        )
        overlay = draw_overlay(page, blocks, selected_id=result["selected_block_id"])
        overlay_width = 720
        display(overlay.resize((overlay_width, int(overlay_width * overlay.height / overlay.width))))
        block_table = pd.DataFrame([
            {
                "id": block["id"],
                "order": block["reading_order"],
                "type": block["type"],
                "area_%": round(100 * area_ratio(block, page), 2),
                "text_preview": block["text"][:180].replace("\n", " "),
            }
            for block in blocks
        ])
        display(block_table.head(20))
    except Exception as exc:
        result["parse_status"] = "error"
        result["warnings"].append(f"parse: {type(exc).__name__}: {exc}")
        print(f"[{label}] Parse failed; continuing: {exc}")
        continue

    if selected is None:
        result["enrichment_status"] = "skipped_no_visual_block"
        result["warnings"].append("No eligible visual block was detected for enrichment.")
        print(f"[{label}] Nano Omni skipped: no eligible visual block.")
        continue

    crop = crop_block(page, selected)
    selected_crops[label] = crop
    display(Markdown(
        f"**Selected crop:** block `{selected['id']}` · class `{selected['type']}` · "
        f"{100 * area_ratio(selected, page):.1f}% of the page"
    ))
    crop_width = min(620, crop.width)
    display(crop.resize((crop_width, int(crop_width * crop.height / crop.width))))

    try:
        started = time.monotonic()
        enrichment, raw_enrichment, usage = call_omni_enrichment(crop)
        result["enrichment_status"] = "ok"
        result["enrichment_seconds"] = round(time.monotonic() - started, 2)
        result["enrichment_usage"] = usage
        result["enrichment"] = enrichment
        result["raw_enrichment_output"] = raw_enrichment
        display(Markdown(
            "**Nano Omni generated visual interpretation — not verbatim OCR**\n\n"
            + (enrichment.get("transcription") or enrichment.get("subject_matter") or "No transcription returned.")
        ))
    except Exception as exc:
        result["enrichment_status"] = "error"
        result["warnings"].append(f"enrichment: {type(exc).__name__}: {exc}")
        print(f"[{label}] enrichment failed; continuing: {exc}")

print("\nModel calls after page pipeline:", CALL_COUNTS)
assert CALL_COUNTS["parse"] <= 5 and CALL_COUNTS["enrichment"] <= 5


## 7. Assemble spatially ordered evidence

Parse text is kept in generated reading order. If a block was enriched, its Nano Omni interpretation is attached to that exact block ID. Page headers carry the stable label, edition, dataset page, and source URL required by the final citation contract.


In [ ]:
def assemble_page_evidence(result: dict[str, Any], *, limit: int = 12000) -> str:
    source = result["source"]
    header = (
        f"===== [{result['label']}] =====\n"
        f"Edition: {source['edition']} ({source['year']})\n"
        f"Dataset page: {source['page']}\n"
        f"Source: {source['ia_page_url']}\n"
        f"Challenge: {source['challenge']}"
    )
    parts = [header]
    selected_id = result.get("selected_block_id")
    enrichment = result.get("enrichment") or {}
    for block in result.get("blocks", []):
        if block.get("text"):
            parts.append(
                f"[Parse 2.0 block {block['id']} · {block['type']}]\n{block['text']}"
            )
        if block["id"] == selected_id and enrichment:
            parts.append(
                "[Nano Omni generated visual interpretation; not verbatim OCR]\n"
                + (enrichment.get("transcription") or enrichment.get("subject_matter") or "")
            )
    joined = "\n\n".join(parts)
    if len(joined) > limit:
        joined = joined[:limit].rsplit("\n", 1)[0] + "\n[page context truncated for quick demo]"
    return joined + f"\n===== END [{result['label']}] ====="


evidence_chunks = [
    assemble_page_evidence(results[p["label"]])
    for p in DEMO_PAGES
    if results[p["label"]].get("parse_status") == "ok"
]
PARSED_PAGE_COUNT = len(evidence_chunks)
RUN_STATUS = "complete" if PARSED_PAGE_COUNT == len(DEMO_PAGES) else "partial"
DOCUMENT_CONTEXT = "\n\n".join(evidence_chunks)
print(
    f"Assembled {PARSED_PAGE_COUNT}/{len(DEMO_PAGES)} pages into "
    f"{len(DOCUMENT_CONTEXT):,} characters · run_status={RUN_STATUS}"
)
print(DOCUMENT_CONTEXT[:3500] + ("\n\n... preview truncated ..." if len(DOCUMENT_CONTEXT) > 3500 else ""))


## 8. Ask one grounded cross-page question

Edit `QUESTION` before this cell's first run. The Reasoning Engine sees the extracted text and generated crop interpretations—not the original page pixels. A deliberate rerun requires resetting the QA counter and makes one additional API call.


In [ ]:
QUESTION = (
    "Compare one Britannica page where an illustration supports dense explanatory text "
    "with one page dominated by a plate or diagram. What different roles do the visuals play?"
)


def call_grounded_qa(question: str, context: str) -> tuple[str, str, dict[str, Any]]:
    if CALL_COUNTS["qa"] >= MAX_QA_CALLS:
        raise RuntimeError(
            "The one-call QA budget is already used. Reset CALL_COUNTS['qa'] to 0 "
            "only if you intentionally want another billed/API call."
        )
    CALL_COUNTS["qa"] += 1
    prompt = f"""\
    Answer the question using only DOCUMENT CONTEXT below.

    Citation and evidence rules:
    1. Cite every substantive claim with one or more stable labels: [B1] through [B5].
    2. Name the cited Britannica edition and dataset page.
    3. Distinguish Parse 2.0 extracted text from Nano Omni generated visual interpretations.
    4. Do not describe a generated visual interpretation as a verbatim quotation or OCR result.
    5. If the context does not support the answer, say exactly: Not supported by the five-page demo.

    DOCUMENT CONTEXT
    {context}

    QUESTION
    {question}
    """
    payload = {
        "model": NANO_OMNI_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 8192,
        "reasoning_budget": 4096,
        "temperature": 0.6,
        "top_p": 0.95,
        "chat_template_kwargs": {"enable_thinking": True},
        "stream": False,
    }
    body = nvidia_completion(payload)
    raw = message_text(body)
    return visible_answer(raw), raw, body.get("usage") or {}


def validate_answer_contract(answer_text: str) -> dict[str, Any]:
    """Check citation/provenance compliance without spending another model call."""
    fallback = "Not supported by the five-page demo."
    if answer_text.strip() == fallback:
        return {
            "compliant": True,
            "unsupported_fallback": True,
            "cited_labels": [],
            "invalid_citations": [],
            "provenance_by_citation": {},
            "evidence_roles_distinguished": True,
        }

    cited_labels = sorted(set(re.findall(r"\[(B[1-5])\]", answer_text)))
    successful_labels = {
        label for label, item in results.items() if item.get("parse_status") == "ok"
    }
    invalid_citations = [label for label in cited_labels if label not in successful_labels]
    provenance_by_citation: dict[str, dict[str, bool]] = {}
    for label in cited_labels:
        source_record = results[label]["source"]
        provenance_by_citation[label] = {
            "edition_present": source_record["edition"].casefold() in answer_text.casefold(),
            "dataset_page_present": bool(re.search(
                rf"\b(?:dataset\s+)?page\s+{source_record['page']}\b",
                answer_text,
                flags=re.IGNORECASE,
            )),
        }

    enrichment_used = any(
        item.get("enrichment_status") == "ok" for item in results.values()
    )
    evidence_roles_distinguished = (
        not enrichment_used
        or ("Parse 2.0" in answer_text and "Nano Omni" in answer_text)
    )
    provenance_ok = bool(cited_labels) and all(
        all(checks.values()) for checks in provenance_by_citation.values()
    )
    return {
        "compliant": bool(
            cited_labels
            and not invalid_citations
            and provenance_ok
            and evidence_roles_distinguished
        ),
        "unsupported_fallback": False,
        "cited_labels": cited_labels,
        "invalid_citations": invalid_citations,
        "provenance_by_citation": provenance_by_citation,
        "evidence_roles_distinguished": evidence_roles_distinguished,
    }


if not DOCUMENT_CONTEXT:
    raise RuntimeError("No successfully parsed page context is available for QA.")

answer, raw_qa_output, qa_usage = call_grounded_qa(QUESTION, DOCUMENT_CONTEXT)
QA_CONTRACT = validate_answer_contract(answer)
display(Markdown(f"**Question:** {QUESTION}\n\n**Grounded answer:**\n\n{answer}"))
if not QA_CONTRACT["compliant"]:
    print("[qa warning] Answer did not fully satisfy the citation/provenance contract:", QA_CONTRACT)
print("\nModel-call ledger:", CALL_COUNTS)


## 9. Export and validation summary

The JSON result stores provenance, timings, model IDs, blocks, warnings, generated text, run completeness, and the answer-contract check. It never stores either secret. The final checks enforce the approved five-page selection and 11-call ceilings without claiming that failed parses succeeded.


In [ ]:
export = {
    "dataset_id": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "parse_model": PARSE_MODEL,
    "omni_model": NANO_OMNI_MODEL,
    "call_counts": dict(CALL_COUNTS),
    "run_status": RUN_STATUS,
    "selected_page_count": len(DEMO_PAGES),
    "parsed_page_count": PARSED_PAGE_COUNT,
    "pages": [results[p["label"]] for p in DEMO_PAGES],
    "question": QUESTION,
    "answer": answer,
    "qa_contract": QA_CONTRACT,
    "raw_qa_output": raw_qa_output,
    "qa_usage": qa_usage,
}

output_path = Path("/content/nemotron_parse_2_britannica_results.json")
output_path.write_text(json.dumps(export, indent=2, ensure_ascii=False), encoding="utf-8")

assert len(export["pages"]) == 5
assert export["call_counts"]["parse"] <= 5
assert export["call_counts"]["enrichment"] <= 5
assert export["call_counts"]["qa"] <= 1
serialized = json.dumps(export)
assert NVIDIA_API_KEY not in serialized and HF_TOKEN not in serialized

summary = pd.DataFrame([
    {
        "label": item["label"],
        "download": item["download_status"],
        "parse": item["parse_status"],
        "enrichment": item["enrichment_status"],
        "blocks": len(item.get("blocks", [])),
        "warnings": len(item.get("warnings", [])),
    }
    for item in export["pages"]
])
display(summary)
print(f"[export] {output_path} ({output_path.stat().st_size / 1024:.1f} KiB)")
print(
    f"[validation] {len(DEMO_PAGES)} selected page records; "
    f"{PARSED_PAGE_COUNT} successful parses; run_status={RUN_STATUS}; "
    "call budgets respected; secrets absent from export."
)
if RUN_STATUS != "complete":
    print("[validation warning] This is a partial run; inspect per-page warnings before using the answer.")
if not QA_CONTRACT["compliant"]:
    print("[validation warning] The generated answer needs citation/provenance repair.")


## 10. What to try next

- Change `QUESTION` and intentionally reset only the QA counter if you want one additional answer.
- Change the Parse prompt to `<predict_text_in_pic>` and compare what moves from Nano Omni back into the parser.
- Replace a manifest row while preserving the `[B#]`, `file_key`, and provenance contract.
- For evaluation, add genuinely human-verified annotations rather than treating the dataset's detector outputs as ground truth.

### Sources and terms

- [Britannica Illustrated Pages dataset card](https://huggingface.co/datasets/biglam/britannica-illustrated-pages)
- [Nemotron Parse 2.0 model card and output format](https://huggingface.co/nvidia/NVIDIA-Nemotron-Parse-2.0)
- [Nemotron Parse 2.0 on build.nvidia.com](https://build.nvidia.com/nvidia/nemotron-parse-2.0)
- [Nemotron 3 Nano Omni on build.nvidia.com](https://build.nvidia.com/nvidia/nemotron-3-nano-omni-30b-a3b-reasoning)

Attribute each scan to its per-row Internet Archive item and contributing library where provided. Dataset curation is published by BigLAM; underlying volumes are public domain in the United States by publication date, as documented by the dataset card. Model outputs remain machine-generated and should be reviewed before high-impact use.
